In [25]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.ops import unary_union
from pyrosm import OSM
from shapely.geometry import Polygon

BASE = "/home/jovyan/work/children15mc"
RAW  = f"{BASE}/data/raw"
PROC = f"{BASE}/data/processed"
FIG  = f"{BASE}/outputs/figures"
LOCAL_PBF = f"{RAW}/greater-london-260812.osm.pbf"

ox.settings.use_cache = True
ox.settings.cache_folder = f"{BASE}/cache"
CRS = 27700

lsoa = gpd.read_file(f"{PROC}/lsoa_london.gpkg")
london = unary_union(lsoa.geometry)
london_4326 = gpd.GeoSeries([london], crs=CRS).to_crs(4326).iloc[0]
print("Configuration complete.")

Configuration complete.


In [26]:
sch = pd.read_csv(f"{RAW}/schools_england.csv", encoding="latin1", low_memory=False)
sch = sch[(sch["PhaseOfEducation (name)"] == "Primary") &
          (sch["EstablishmentStatus (name)"] == "Open") &
          (sch["Easting"].notna())].copy()
schools = gpd.GeoDataFrame(
    sch, geometry=gpd.points_from_xy(sch["Easting"], sch["Northing"]), crs=CRS)
schools = schools[schools.within(london)][["EstablishmentName", "geometry"]]
print(f"London Primary School: {len(schools)}")
schools.to_file(f"{PROC}/schools_london.gpkg", driver="GPKG")

London Primary School: 1752


In [27]:
import os, glob, time
from pyrosm import OSM

# 1
osm = OSM(LOCAL_PBF)
parks_data = osm.get_data_by_custom_criteria(custom_filter={"leisure": ["park"]})
parks = parks_data[parks_data.geometry.type.isin(["Polygon", "MultiPolygon"])].to_crs(CRS)
parks = parks.reset_index(drop=True).reset_index().rename(columns={"index": "park_id"})
parks = parks[["park_id", "geometry"]]
print(f"Extracted London park polygons: {len(parks)} parks")

# 2
gates_data = osm.get_data_by_custom_criteria(custom_filter={
    "barrier": ["gate"], 
    "entrance": ["yes", "main"]
})
entry_pts = gates_data[gates_data.geometry.type == "Point"].to_crs(CRS)
print(f"Successfully extracted entrances/gates: {len(entry_pts)}")

# 3
park_buf = parks.copy()
park_buf["geometry"] = park_buf.geometry.boundary.buffer(20)
matched = gpd.sjoin(entry_pts, park_buf[["park_id", "geometry"]],
                    how="inner", predicate="within")[["park_id", "geometry"]].drop_duplicates()
n_tagged = matched["park_id"].nunique()
print(f"Parks with tagged entrances: {n_tagged} ({100*n_tagged/len(parks):.1f}%)")

# 4
have = set(matched["park_id"])
missing = parks[~parks["park_id"].isin(have)]
print(f"Parks requiring generated entrances: {len(missing)}")

rows = []
for _, r in missing.iterrows():
    bnd = r.geometry.boundary
    lines = [bnd] if bnd.geom_type == "LineString" else list(bnd.geoms)
    for ln in lines:
        n = max(int(ln.length // 25), 1)
        for i in range(n):
            rows.append({"park_id": r["park_id"], "geometry": ln.interpolate(i * 25)})

sampled = gpd.GeoDataFrame(rows, crs=CRS) if rows else gpd.GeoDataFrame(geometry=[], crs=CRS)
park_entries = gpd.GeoDataFrame(pd.concat([matched, sampled], ignore_index=True), crs=CRS)

print(f"\nTotal park entry nodes for calculation: {len(park_entries)}")
park_entries.to_file(f"{PROC}/park_entries.gpkg", driver="GPKG")

Extracted London park polygons: 3250 parks
Successfully extracted entrances/gates: 72853
Parks with tagged entrances: 1697 (52.2%)
Parks requiring generated entrances: 1553

Total park entry nodes for calculation: 30791


In [28]:
def count_facilities(iso_gdf, schools, park_entries):
    
    js = gpd.sjoin(schools, iso_gdf, how="inner", predicate="within")
    n_sch = js.groupby("lsoa21cd").size().rename("n_schools")

    jp = gpd.sjoin(park_entries, iso_gdf, how="inner", predicate="within")
    n_park = jp.groupby("lsoa21cd")["park_id"].nunique().rename("n_parks")

    out = iso_gdf[["lsoa21cd"]].copy()
    out = out.merge(n_sch, on="lsoa21cd", how="left")
    out = out.merge(n_park, on="lsoa21cd", how="left")
    out[["n_schools", "n_parks"]] = out[["n_schools", "n_parks"]].fillna(0).astype(int)
    out["score"] = out["n_schools"] + out["n_parks"]
    return out

In [29]:
iso_d = gpd.read_file(f"{PROC}/isochrones_all.gpkg")
iso_b = gpd.read_file(f"{PROC}/isochrones_barrier.gpkg")

acc_d = count_facilities(iso_d, schools, park_entries)
acc_b = count_facilities(iso_b, schools, park_entries)

print("Default network")
print(acc_d[["n_schools","n_parks","score"]].describe().round(2))
print("Barrier network")
print(acc_b[["n_schools","n_parks","score"]].describe().round(2))

Default network
       n_schools  n_parks    score
count    4994.00  4994.00  4994.00
mean        2.31     4.82     7.13
std         1.66     4.90     5.88
min         0.00     0.00     0.00
25%         1.00     2.00     3.00
50%         2.00     3.00     6.00
75%         3.00     6.00     9.00
max        11.00    36.00    43.00
Barrier network
       n_schools  n_parks    score
count    4964.00  4964.00  4964.00
mean        1.35     2.94     4.29
std         1.41     3.79     4.75
min         0.00     0.00     0.00
25%         0.00     0.00     1.00
50%         1.00     2.00     3.00
75%         2.00     4.00     6.00
max         8.00    28.00    35.00


In [30]:
cmp = acc_d.rename(columns={"n_schools":"sch_d","n_parks":"park_d","score":"score_d"}) \
           .merge(acc_b.rename(columns={"n_schools":"sch_b","n_parks":"park_b","score":"score_b"}),
                  on="lsoa21cd", how="inner")

cmp["loss_abs"] = cmp["score_d"] - cmp["score_b"]
cmp["loss_pct"] = np.where(cmp["score_d"] > 0,
                           100 * cmp["loss_abs"] / cmp["score_d"], np.nan)

print(f"Matched LSOAs: {len(cmp)}")
print(f"Avg accessible facilities: Default {cmp.score_d.mean():.2f} → Barrier {cmp.score_b.mean():.2f}")
print(f"Average loss: {cmp.loss_abs.mean():.2f} ({cmp.loss_pct.mean():.1f}%)")
print(f"Median loss percentage: {cmp.loss_pct.median():.1f}%")
print()
print("Key comparisons:")
for k in [1, 2]:
    a = 100*(cmp.score_d >= k).mean()
    b = 100*(cmp.score_b >= k).mean()
    print(f"  LSOAs with ≥{k} accessible facilities: Default {a:.1f}% → Barrier {b:.1f}%")
print(f"  LSOAs with 0 accessible facilities: Default {100*(cmp.score_d==0).mean():.1f}% "
      f"→ Barrier {100*(cmp.score_b==0).mean():.1f}%")

df = gpd.read_file(f"{PROC}/analysis_table.gpkg")
df = df.merge(cmp, on="lsoa21cd", how="left")
df.to_file(f"{PROC}/accessibility_comparison.gpkg", driver="GPKG")
print("\n✓ Saved accessibility_comparison.gpkg")

Matched LSOAs: 4964
Avg accessible facilities: Default 7.14 → Barrier 4.29
Average loss: 2.85 (42.7%)
Median loss percentage: 39.1%

Key comparisons:
  LSOAs with ≥1 accessible facilities: Default 98.1% → Barrier 81.9%
  LSOAs with ≥2 accessible facilities: Default 92.6% → Barrier 67.4%
  LSOAs with 0 accessible facilities: Default 1.9% → Barrier 18.1%

✓ Saved accessibility_comparison.gpkg
